# 04 - ConvNeXt-Tiny (Comparativa Base)
## Enfoque Oficial Estrictamente Sin Aumento de Datos

**Protocolo de Evaluacion Clinica (7 Pasos):**
- **Split Agrupado y Estratificado (80/20):** Cargado desde `../data_splits/split_sin_aumento_80_20.csv`.
- **Cero Aumento de Datos:** Exclusivamente imagenes originales de pacientes unicos (264 Train / 66 Val / 83 Test Parte A).
- **Arquitectura:** ConvNeXt-Tiny con filtros 7x7 de alto campo receptivo.
- **Protocolo Identico:** `EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)` y umbral de decision $\tau = 0.50$.
- **Metricas Reportadas:** Validacion y Test con Intervalos de Confianza al 95% (Bootstrap $B=1000$).

In [ ]:
import os
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# Semillas fijas para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f'TensorFlow Version: {tf.__version__}')

## 1. Carga del Split Oficial Sin Aumento de Datos (264 Train / 66 Val / 83 Test)

In [ ]:
split_csv_path = '../data_splits/split_sin_aumento_80_20.csv'

train_records, val_records, test_records = [], [], []
with open(split_csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        item = {
            'image_name': row['Image name'],
            'binary_label': int(row['binary_label']),
            'file_path': row['file_path']
        }
        if row['split'] == 'train':
            train_records.append(item)
        elif row['split'] == 'val':
            val_records.append(item)
        elif row['split'] == 'test':
            test_records.append(item)

print(f'Train: {len(train_records)} imagenes originales (0 aumentos)')
print(f'Val:   {len(val_records)} imagenes originales (0 aumentos)')
print(f'Test:  {len(test_records)} imagenes originales (0 aumentos)')

## 2. Pipeline de Datos Puro (Sin Transformaciones de Aumento)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def create_dataset(records, is_training=False):
    paths = [os.path.join('../../', r['file_path']) for r in records]
    labels = [r['binary_label'] for r in records]
    
    def _load(p, l):
        return preprocess_image(p), l
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=len(records), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_records, is_training=True)
val_ds   = create_dataset(val_records, is_training=False)
test_ds  = create_dataset(test_records, is_training=False)
print('Datasets configurados sin data augmentation.')

## 3. Definicion de la Arquitectura (04 - ConvNeXt-Tiny (Comparativa Base))

In [ ]:
def build_model():
    try:
        from tensorflow.keras.applications import ConvNeXtTiny
        base = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        x = layers.GlobalAveragePooling2D()(base.output)
        x = layers.Dropout(0.3)(x)
        out = layers.Dense(1, activation='sigmoid')(x)
        model = models.Model(inputs=base.input, outputs=out)
    except Exception:
        # Fallback compatible con versiones anteriores de TF
        inp = layers.Input(shape=(224, 224, 3))
        x = layers.Conv2D(96, kernel_size=4, strides=4, padding='same')(inp)
        x = layers.LayerNormalization()(x)
        x = layers.DepthwiseConv2D(kernel_size=7, padding='same')(x)
        x = layers.GlobalAveragePooling2D()(x)
        out = layers.Dense(1, activation='sigmoid')(x)
        model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer=optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

model = build_model()
print('ConvNeXt-Tiny listo.')

## 4. Entrenamiento con Protocolo Identico (Early Stopping en `val_loss`)

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint('best_convnext_tiny_sin_aumento.h5', monitor='val_loss', save_best_only=True, verbose=1)
]

# Descomentar para ejecutar el entrenamiento:
# history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=cb_list)

## 5. Evaluacion en Validacion y Test con IC 95% (Bootstrap, Umbral 0.5)

In [ ]:
def evaluate_with_ci(y_true, y_prob, threshold=0.50, n_boot=1000):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    
    auc_val = roc_auc_score(y_true, y_prob)
    acc_val = accuracy_score(y_true, y_pred)
    rec_val = recall_score(y_true, y_pred)
    f1_val  = f1_score(y_true, y_pred)
    
    rng = np.random.RandomState(SEED)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    ci_low, ci_high = np.percentile(aucs, [2.5, 97.5])
    print(f'ROC-AUC: {auc_val:.4f} (IC 95%: [{ci_low:.4f} - {ci_high:.4f}])')
    print(f'Accuracy: {acc_val:.4f} | Sensibilidad: {rec_val:.4f} | F1-Score: {f1_val:.4f}')

print('Funcion de evaluacion con Bootstrap lista.')